In [1]:
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from navigoquest.utils.config import standard_dirs, LEVELS, METADATA_COLS, AGE_RANGE
from navigoquest import format_metric_task_data
from navigoquest.utils.pipelines import load_environment, load_paths_normative, pipeline_compute_features

In [2]:
# Settings common over levels
early_stop = np.inf
use_sparse_norm = True
output_filenames_old = {lvl: f"metrics_level{lvl:02}_gb_2480.csv" for lvl in np.arange(1, 99)}
output_filenames = {lvl: f"metrics_level{lvl:02}_gb_2480.csv" for lvl in np.arange(1, 99)}
task_list = ['voc', 'path_length', 'average_curvature', 'boundary_affinity',
             'frobenius_deviation', 'supremum_deviation', 'conformity', 'vector_conformity']
levels_now = LEVELS

# Directories
dirs = standard_dirs()
paths_dir = dirs['paths']
output_dir = dirs['output']
if not output_dir.exists():
    output_dir.mkdir(parents=True)

# Load environment
env_store = load_environment(dirs)

Loading environment data...
done. 



# Benchmark data

In [3]:
# Load dataframes this way
output_df_all_loaded = {lvl: pd.read_csv(output_dir / output_filenames[lvl]) for lvl in levels_now}

# Restrict to age range 50 to 80
output_df_all_5080 = {}
for lvl in LEVELS:
    item = output_df_all_loaded[lvl]
    age_filter = (item['age'] >= 50) & (item['age'] <= 80)
    output_df_all_5080[lvl] = item[age_filter]

In [4]:
# Entry counts for two genders
gender_counts = {}
for lvl in LEVELS:
    item_genders = output_df_all_5080[lvl]['gender']
    item_female_count = np.sum((item_genders == 'f').to_numpy())
    item_male_count = np.sum((item_genders == 'm').to_numpy())
    assert len(item_genders) == (item_female_count + item_male_count)
    female_ratio = item_female_count / len(item_genders)

    gender_counts[lvl] = {'f': item_female_count, 'm': item_male_count, 'female_ratio': female_ratio}

In [5]:
print(f'Total counts: ')
print([len(output_df_all_5080[lvl]) for lvl in LEVELS])

print(f'Female counts: ')
print([int(gender_counts[lvl]['f']) for lvl in LEVELS])

print(f'Male counts: ')
print([int(gender_counts[lvl]['m']) for lvl in LEVELS])

print(f'Female ratios: ')
print([round(float(gender_counts[lvl]['female_ratio']), 2) for lvl in LEVELS])

Total counts: 
[75494, 74094, 60976, 54832, 45113]
Female counts: 
[40328, 39530, 31843, 27962, 22623]
Male counts: 
[35166, 34564, 29133, 26870, 22490]
Female ratios: 
[0.53, 0.53, 0.52, 0.51, 0.5]


# Clinical data

In [6]:
# Load clinical dataframe
clinical_df_loaded = pd.read_csv(output_dir / "clinical_metrics.csv")
clinical_subgroups = ['ad', 'e3e3', 'e3e4', 'e4e4']

In [7]:
# Entry counts for two genders (Clinical)
gender_counts_clinical = {gp: {} for gp in clinical_subgroups}
for gp in clinical_subgroups:
    for lvl in LEVELS:
        # Isolate the subgroup of interest with the level
        mask = (clinical_df_loaded['level'] == lvl) & (clinical_df_loaded['group'] == gp)

        # Count genders
        item_genders = clinical_df_loaded[mask]['gender']
        item_female_count = np.sum((item_genders == 'f').to_numpy())
        item_male_count = np.sum((item_genders == 'm').to_numpy())
        assert len(item_genders) == (item_female_count + item_male_count)
        female_ratio = item_female_count / len(item_genders)
        gender_counts_clinical[gp][lvl] = {'f': item_female_count, 'm': item_male_count, 'female_ratio': female_ratio}


In [8]:
gp = 'ad'

print(f'Gender statistics for clinical subgroup "{gp}" \n')
print(f'Total counts: ')
print([int(gender_counts_clinical[gp][lvl]['f'] + gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female counts: ')
print([int(gender_counts_clinical[gp][lvl]['f']) for lvl in LEVELS])
print(f'Male counts: ')
print([int(gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female ratios: ')
print([round(float(gender_counts_clinical[gp][lvl]['female_ratio']), 2) for lvl in LEVELS])

Gender statistics for clinical subgroup "ad" 

Total counts: 
[21, 20, 18, 15, 11]
Female counts: 
[12, 11, 10, 7, 5]
Male counts: 
[9, 9, 8, 8, 6]
Female ratios: 
[0.57, 0.55, 0.56, 0.47, 0.45]


In [9]:
gp = 'e3e3'

print(f'Gender statistics for clinical subgroup "{gp}" \n')
print(f'Total counts: ')
print([int(gender_counts_clinical[gp][lvl]['f'] + gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female counts: ')
print([int(gender_counts_clinical[gp][lvl]['f']) for lvl in LEVELS])
print(f'Male counts: ')
print([int(gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female ratios: ')
print([round(float(gender_counts_clinical[gp][lvl]['female_ratio']), 2) for lvl in LEVELS])

Gender statistics for clinical subgroup "e3e3" 

Total counts: 
[28, 28, 28, 28, 28]
Female counts: 
[15, 15, 15, 15, 15]
Male counts: 
[13, 13, 13, 13, 13]
Female ratios: 
[0.54, 0.54, 0.54, 0.54, 0.54]


In [10]:
gp = 'e3e4'

print(f'Gender statistics for clinical subgroup "{gp}" \n')
print(f'Total counts: ')
print([int(gender_counts_clinical[gp][lvl]['f'] + gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female counts: ')
print([int(gender_counts_clinical[gp][lvl]['f']) for lvl in LEVELS])
print(f'Male counts: ')
print([int(gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female ratios: ')
print([round(float(gender_counts_clinical[gp][lvl]['female_ratio']), 2) for lvl in LEVELS])

Gender statistics for clinical subgroup "e3e4" 

Total counts: 
[31, 31, 31, 31, 31]
Female counts: 
[11, 11, 11, 11, 11]
Male counts: 
[20, 20, 20, 20, 20]
Female ratios: 
[0.35, 0.35, 0.35, 0.35, 0.35]


In [11]:
gp = 'e4e4'

print(f'Gender statistics for clinical subgroup "{gp}" \n')
print(f'Total counts: ')
print([int(gender_counts_clinical[gp][lvl]['f'] + gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female counts: ')
print([int(gender_counts_clinical[gp][lvl]['f']) for lvl in LEVELS])
print(f'Male counts: ')
print([int(gender_counts_clinical[gp][lvl]['m']) for lvl in LEVELS])
print(f'Female ratios: ')
print([round(float(gender_counts_clinical[gp][lvl]['female_ratio']), 2) for lvl in LEVELS])

Gender statistics for clinical subgroup "e4e4" 

Total counts: 
[5, 5, 5, 5, 5]
Female counts: 
[3, 3, 3, 3, 3]
Male counts: 
[2, 2, 2, 2, 2]
Female ratios: 
[0.6, 0.6, 0.6, 0.6, 0.6]


# Visiting order correctness

In [12]:
for lvl in [6, 8, 11]:
    voc_now = output_df_all_5080[lvl]['voc'].to_numpy()
    voc1_prop = np.sum(voc_now) / voc_now.size
    print(f'Level {lvl}: prop. of correct visiting order = {round(voc1_prop*100, 5)}%.')

Level 6: prop. of correct visiting order = 80.88592%.
Level 8: prop. of correct visiting order = 75.65837%.
Level 11: prop. of correct visiting order = 79.58238%.


# Age distribution

In [13]:
# Mean age ± SD
groups = ['ad', 'e3e3', 'e3e4']
levels_use = [6, 8, 11]

rows = []
for lvl in levels_use:
    ages_bench = output_df_all_5080[lvl]['age'].to_numpy()
    bench_str = f"{np.mean(ages_bench):.2f} ± {np.std(ages_bench):.2f}"
    row = [f"Level {lvl}", bench_str]
    for gp in groups:
        mask = (clinical_df_loaded['level'] == lvl) & (clinical_df_loaded['group'] == gp)
        ages = clinical_df_loaded[mask]['age'].to_numpy()
        row.append(f"{np.mean(ages):.2f} ± {np.std(ages):.2f}")
    rows.append(row)

df_display = pd.DataFrame(rows, columns=['', 'Benchmark', 'AD', 'e3e3', 'e3e4'])
df_display = df_display.set_index('')
print(df_display.to_string())

             Benchmark            AD          e3e3          e3e4
                                                                
Level 6   60.12 ± 7.04  72.39 ± 6.07  62.25 ± 5.82  62.16 ± 6.63
Level 8   59.96 ± 6.96  72.07 ± 6.48  62.25 ± 5.82  62.16 ± 6.63
Level 11  59.86 ± 6.88  71.45 ± 7.00  62.25 ± 5.82  62.16 ± 6.63


In [14]:
# Median age and range
groups = ['ad', 'e3e3', 'e3e4']

rows = []
for lvl in levels_use:
    ages_bench = output_df_all_5080[lvl]['age'].to_numpy()
    bench_str = f"{np.median(ages_bench):.1f}, ({np.min(ages_bench):.0f}, {np.max(ages_bench):.0f})"
    row = [f"Level {lvl}", bench_str]
    for gp in groups:
        mask = (clinical_df_loaded['level'] == lvl) & (clinical_df_loaded['group'] == gp)
        ages = clinical_df_loaded[mask]['age'].to_numpy()
        row.append(f"{np.median(ages):.1f}, ({np.min(ages):.0f}, {np.max(ages):.0f})")
    rows.append(row)

df_display = pd.DataFrame(rows, columns=['', 'Benchmark', 'AD', 'e3e3', 'e3e4'])
df_display = df_display.set_index('')
print(df_display.to_string())

               Benchmark              AD            e3e3            e3e4
                                                                        
Level 6   59.0, (50, 80)  73.0, (53, 79)  60.5, (53, 72)  62.0, (51, 74)
Level 8   59.0, (50, 80)  73.0, (53, 79)  60.5, (53, 72)  62.0, (51, 74)
Level 11  59.0, (50, 80)  73.0, (53, 78)  60.5, (53, 72)  62.0, (51, 74)
